In [23]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [5]:
CONFIG = {
    "mnist": {
        "batch_size": 128,
        "epochs": 10,
    },

    "cifar10": {
        "batch_size": 128,
        "epochs": 30,
    }
}

In [6]:
from pathlib import Path

ROOT = Path.cwd()

# If the notebook is running from the notebooks/ directory
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)

Project root: /home/mahdi/Documents/cross-norm-adversarial-robustness
Data directory: /home/mahdi/Documents/cross-norm-adversarial-robustness/data


In [7]:
mnist_train_transform = transforms.Compose([
    transforms.ToTensor(),
])

mnist_test_transform = transforms.Compose([
    transforms.ToTensor(),
])


cifar_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

cifar_test_transform = transforms.Compose([
    transforms.ToTensor(),
])

In [8]:
mnist_train_dataset = torchvision.datasets.MNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=mnist_train_transform,
)

mnist_test_dataset = torchvision.datasets.MNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=mnist_test_transform,
)


cifar_train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=cifar_train_transform,
)

cifar_test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=cifar_test_transform,
)

100.0%
100.0%
100.0%
100.0%
100.0%


In [9]:
print("MNIST train:", len(mnist_train_dataset))
print("MNIST test :", len(mnist_test_dataset))

print("CIFAR-10 train:", len(cifar_train_dataset))
print("CIFAR-10 test :", len(cifar_test_dataset))

MNIST train: 60000
MNIST test : 10000
CIFAR-10 train: 50000
CIFAR-10 test : 10000


In [10]:
mnist_train_loader = DataLoader(
    mnist_train_dataset,
    batch_size=CONFIG["mnist"]["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

mnist_test_loader = DataLoader(
    mnist_test_dataset,
    batch_size=CONFIG["mnist"]["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


cifar_train_loader = DataLoader(
    cifar_train_dataset,
    batch_size=CONFIG["cifar10"]["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

cifar_test_loader = DataLoader(
    cifar_test_dataset,
    batch_size=CONFIG["cifar10"]["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

In [14]:
mnist_images, mnist_labels = next(iter(mnist_train_loader))
cifar_images, cifar_labels = next(iter(cifar_train_loader))

print("MNIST")
print("Images shape:", mnist_images.shape)
print("Labels shape:", mnist_labels.shape)
print("Pixel range:", mnist_images.min().item(), mnist_images.max().item())

print()

print("CIFAR-10")
print("Images shape:", cifar_images.shape)
print("Labels shape:", cifar_labels.shape)
print("Pixel range:", cifar_images.min().item(), cifar_images.max().item())

MNIST
Images shape: torch.Size([128, 1, 28, 28])
Labels shape: torch.Size([128])
Pixel range: 0.0 1.0

CIFAR-10
Images shape: torch.Size([128, 3, 32, 32])
Labels shape: torch.Size([128])
Pixel range: 0.0 1.0


In [15]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 14 * 14, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [16]:
mnist_model = MNISTCNN().to(device)

print(mnist_model)

MNISTCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=12544, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [17]:
dummy_input = torch.randn(8, 1, 28, 28).to(device)

dummy_output = mnist_model(dummy_input)

print("Input shape :", dummy_input.shape)
print("Output shape:", dummy_output.shape)

Input shape : torch.Size([8, 1, 28, 28])
Output shape: torch.Size([8, 10])


In [18]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [19]:
def evaluate_clean(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            running_loss += loss.item() * images.size(0)

            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [20]:
def pgd_linf_attack(
    model,
    images,
    labels,
    epsilon,
    alpha,
    steps,
    random_start=True
):
    original_images = images.detach()

    if random_start:
        delta = torch.empty_like(original_images).uniform_(
            -epsilon, epsilon
        )
        adversarial_images = original_images + delta
        adversarial_images = torch.clamp(
            adversarial_images, 0.0, 1.0
        )
    else:
        adversarial_images = original_images.clone()

    for _ in range(steps):
        adversarial_images.requires_grad_(True)

        logits = model(adversarial_images)
        loss = F.cross_entropy(logits, labels)

        gradient = torch.autograd.grad(
            loss,
            adversarial_images
        )[0]

        adversarial_images = (
            adversarial_images.detach()
            + alpha * gradient.sign()
        )

        delta = adversarial_images - original_images
        delta = torch.clamp(
            delta,
            min=-epsilon,
            max=epsilon
        )

        adversarial_images = torch.clamp(
            original_images + delta,
            0.0,
            1.0
        ).detach()

    return adversarial_images

In [ ]:
def pgd_l2_attack(
    model,
    images,
    labels,
    epsilon,
    alpha,
    steps,
    random_start=True
):
    original_images = images.detach()

    if random_start:
        delta = torch.randn_like(original_images)

        delta_flat = delta.view(delta.size(0), -1)
        delta_norm = delta_flat.norm(p=2, dim=1, keepdim=True)

        delta_flat = delta_flat / (delta_norm + 1e-12)

        dimensions = delta_flat.size(1)

        radius = torch.rand(
            delta.size(0),
            1,
            device=delta.device
        ).pow(1.0 / dimensions)

        delta_flat = delta_flat * radius * epsilon
        delta = delta_flat.view_as(original_images)

        adversarial_images = torch.clamp(
            original_images + delta,
            0.0,
            1.0
        )

    else:
        adversarial_images = original_images.clone()

    for _ in range(steps):
        adversarial_images.requires_grad_(True)

        logits = model(adversarial_images)
        loss = F.cross_entropy(logits, labels)

        gradient = torch.autograd.grad(
            loss,
            adversarial_images
        )[0]

        gradient_flat = gradient.view(gradient.size(0), -1)

        gradient_norm = gradient_flat.norm(
            p=2,
            dim=1,
            keepdim=True
        )

        normalized_gradient = (
            gradient_flat / (gradient_norm + 1e-12)
        ).view_as(gradient)

        adversarial_images = (
            adversarial_images.detach()
            + alpha * normalized_gradient
        )

        delta = adversarial_images - original_images

        delta_flat = delta.view(delta.size(0), -1)

        delta_norm = delta_flat.norm(
            p=2,
            dim=1,
            keepdim=True
        )

        projection_factor = torch.clamp(
            epsilon / (delta_norm + 1e-12),
            max=1.0
        )

        delta_flat = delta_flat * projection_factor
        delta = delta_flat.view_as(original_images)

        adversarial_images = torch.clamp(
            original_images + delta,
            0.0,
            1.0
        ).detach()

    return adversarial_images

In [ ]:
ATTACK_CONFIG = {
    "mnist": {
        "linf": {
            "train_epsilon": 0.30,
            "train_alpha": 0.01,
            "train_steps": 40,
        },

        "l2": {
            "train_epsilon": 2.0,
            "train_alpha": 0.10,
            "train_steps": 40,
        },
    },

    "cifar10": {
        "linf": {
            "train_epsilon": 8 / 255,
            "train_alpha": 2 / 255,
            "train_steps": 10,
        },

        "l2": {
            "train_epsilon": 0.5,
            "train_alpha": 0.05,
            "train_steps": 10,
        },
    }
}

In [ ]:
images, labels = next(iter(mnist_test_loader))

images = images.to(device)
labels = labels.to(device)

adversarial_images = pgd_linf_attack(
    model=mnist_model,
    images=images,
    labels=labels,
    epsilon=0.3,
    alpha=0.01,
    steps=40,
    random_start=True
)

perturbation = adversarial_images - images

print(
    "Maximum L_inf perturbation:",
    perturbation.abs().max().item()
)

print(
    "Adversarial pixel range:",
    adversarial_images.min().item(),
    adversarial_images.max().item()
)

In [ ]:
images, labels = next(iter(mnist_test_loader))

images = images.to(device)
labels = labels.to(device)

adversarial_images = pgd_l2_attack(
    model=mnist_model,
    images=images,
    labels=labels,
    epsilon=2.0,
    alpha=0.1,
    steps=40,
    random_start=True
)

perturbation = adversarial_images - images

perturbation_flat = perturbation.view(
    perturbation.size(0),
    -1
)

l2_norms = perturbation_flat.norm(
    p=2,
    dim=1
)

print("Maximum L2 perturbation:", l2_norms.max().item())

print(
    "Adversarial pixel range:",
    adversarial_images.min().item(),
    adversarial_images.max().item()
)

In [ ]:
def train_adversarial_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    attack_fn,
    attack_kwargs
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # Generate adversarial examples using the current model
        adversarial_images = attack_fn(
            model=model,
            images=images,
            labels=labels,
            **attack_kwargs
        )

        optimizer.zero_grad()

        logits = model(adversarial_images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
base_mnist_model = MNISTCNN()

initial_state = copy.deepcopy(
    base_mnist_model.state_dict()
)

mnist_clean_model = MNISTCNN().to(device)
mnist_linf_model = MNISTCNN().to(device)
mnist_l2_model = MNISTCNN().to(device)

mnist_clean_model.load_state_dict(initial_state)
mnist_linf_model.load_state_dict(initial_state)
mnist_l2_model.load_state_dict(initial_state)

In [ ]:
mnist_clean_optimizer = torch.optim.Adam(
    mnist_clean_model.parameters(),
    lr=1e-3
)

mnist_linf_optimizer = torch.optim.Adam(
    mnist_linf_model.parameters(),
    lr=1e-3
)

mnist_l2_optimizer = torch.optim.Adam(
    mnist_l2_model.parameters(),
    lr=1e-3
)

In [ ]:
mnist_linf_train_attack = {
    "epsilon": ATTACK_CONFIG["mnist"]["linf"]["train_epsilon"],
    "alpha": ATTACK_CONFIG["mnist"]["linf"]["train_alpha"],
    "steps": ATTACK_CONFIG["mnist"]["linf"]["train_steps"],
    "random_start": True,
}

mnist_l2_train_attack = {
    "epsilon": ATTACK_CONFIG["mnist"]["l2"]["train_epsilon"],
    "alpha": ATTACK_CONFIG["mnist"]["l2"]["train_alpha"],
    "steps": ATTACK_CONFIG["mnist"]["l2"]["train_steps"],
    "random_start": True,
}

In [ ]:
mnist_clean_history = []

for epoch in range(CONFIG["mnist"]["epochs"]):
    train_loss, train_acc = train_one_epoch(
        model=mnist_clean_model,
        loader=mnist_train_loader,
        optimizer=mnist_clean_optimizer,
        criterion=criterion,
        device=device
    )

    test_loss, test_acc = evaluate_clean(
        model=mnist_clean_model,
        loader=mnist_test_loader,
        criterion=criterion,
        device=device
    )

    mnist_clean_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
    })

    print(
        f"[Clean] Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc * 100:.2f}% | "
        f"Test Acc: {test_acc * 100:.2f}%"
    )

In [ ]:
mnist_linf_history = []

for epoch in range(CONFIG["mnist"]["epochs"]):
    train_loss, train_acc = train_adversarial_one_epoch(
        model=mnist_linf_model,
        loader=mnist_train_loader,
        optimizer=mnist_linf_optimizer,
        criterion=criterion,
        device=device,
        attack_fn=pgd_linf_attack,
        attack_kwargs=mnist_linf_train_attack
    )

    clean_test_loss, clean_test_acc = evaluate_clean(
        model=mnist_linf_model,
        loader=mnist_test_loader,
        criterion=criterion,
        device=device
    )

    mnist_linf_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_adversarial_accuracy": train_acc,
        "clean_test_loss": clean_test_loss,
        "clean_test_accuracy": clean_test_acc,
    })

    print(
        f"[Linf-AT] Epoch {epoch + 1:02d} | "
        f"Adv Train Loss: {train_loss:.4f} | "
        f"Adv Train Acc: {train_acc * 100:.2f}% | "
        f"Clean Test Acc: {clean_test_acc * 100:.2f}%"
    )

In [ ]:
mnist_l2_history = []

for epoch in range(CONFIG["mnist"]["epochs"]):
    train_loss, train_acc = train_adversarial_one_epoch(
        model=mnist_l2_model,
        loader=mnist_train_loader,
        optimizer=mnist_l2_optimizer,
        criterion=criterion,
        device=device,
        attack_fn=pgd_l2_attack,
        attack_kwargs=mnist_l2_train_attack
    )

    clean_test_loss, clean_test_acc = evaluate_clean(
        model=mnist_l2_model,
        loader=mnist_test_loader,
        criterion=criterion,
        device=device
    )

    mnist_l2_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_adversarial_accuracy": train_acc,
        "clean_test_loss": clean_test_loss,
        "clean_test_accuracy": clean_test_acc,
    })

    print(
        f"[L2-AT] Epoch {epoch + 1:02d} | "
        f"Adv Train Loss: {train_loss:.4f} | "
        f"Adv Train Acc: {train_acc * 100:.2f}% | "
        f"Clean Test Acc: {clean_test_acc * 100:.2f}%"
    )

In [ ]:
CHECKPOINT_DIR = ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    mnist_clean_model.state_dict(),
    CHECKPOINT_DIR / "mnist_clean.pt"
)

torch.save(
    mnist_linf_model.state_dict(),
    CHECKPOINT_DIR / "mnist_pgd_linf.pt"
)

torch.save(
    mnist_l2_model.state_dict(),
    CHECKPOINT_DIR / "mnist_pgd_l2.pt"
)

In [ ]:
def evaluate_under_attack(
    model,
    loader,
    attack_fn,
    attack_kwargs,
    device,
    restarts=1
):
    model.eval()

    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # Track the highest loss found for each sample
        worst_loss = torch.full(
            (images.size(0),),
            -float("inf"),
            device=device
        )

        worst_adversarial_images = images.clone()

        for _ in range(restarts):
            adversarial_images = attack_fn(
                model=model,
                images=images,
                labels=labels,
                **attack_kwargs
            )

            with torch.no_grad():
                logits = model(adversarial_images)

                losses = F.cross_entropy(
                    logits,
                    labels,
                    reduction="none"
                )

            stronger_mask = losses > worst_loss

            worst_loss[stronger_mask] = losses[stronger_mask]

            worst_adversarial_images[stronger_mask] = (
                adversarial_images[stronger_mask]
            )

        with torch.no_grad():
            logits = model(worst_adversarial_images)

        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return correct / total

In [ ]:
TEST_ATTACK_CONFIG = {
    "mnist": {
        "linf": {
            "epsilon": 0.30,
            "alpha": 0.01,
            "steps": 50,
            "random_start": True,
        },

        "l2": {
            "epsilon": 2.0,
            "alpha": 0.10,
            "steps": 50,
            "random_start": True,
        },
    },

    "cifar10": {
        "linf": {
            "epsilon": 8 / 255,
            "alpha": 2 / 255,
            "steps": 20,
            "random_start": True,
        },

        "l2": {
            "epsilon": 0.5,
            "alpha": 0.05,
            "steps": 20,
            "random_start": True,
        },
    }
}

In [ ]:
mnist_models = {
    "clean": mnist_clean_model,
    "linf_at": mnist_linf_model,
    "l2_at": mnist_l2_model,
}

mnist_attacks = {
    "pgd_linf": (
        pgd_linf_attack,
        TEST_ATTACK_CONFIG["mnist"]["linf"]
    ),

    "pgd_l2": (
        pgd_l2_attack,
        TEST_ATTACK_CONFIG["mnist"]["l2"]
    ),
}

In [ ]:
mnist_cross_norm_results = []

for model_name, model in mnist_models.items():

    clean_loss, clean_accuracy = evaluate_clean(
        model=model,
        loader=mnist_test_loader,
        criterion=criterion,
        device=device
    )

    mnist_cross_norm_results.append({
        "model": model_name,
        "attack": "clean",
        "epsilon": 0.0,
        "accuracy": clean_accuracy,
    })

    for attack_name, (attack_fn, attack_kwargs) in mnist_attacks.items():

        adversarial_accuracy = evaluate_under_attack(
            model=model,
            loader=mnist_test_loader,
            attack_fn=attack_fn,
            attack_kwargs=attack_kwargs,
            device=device
        )

        mnist_cross_norm_results.append({
            "model": model_name,
            "attack": attack_name,
            "epsilon": attack_kwargs["epsilon"],
            "accuracy": adversarial_accuracy,
        })

        print(
            f"{model_name:8s} | "
            f"{attack_name:10s} | "
            f"Accuracy: {adversarial_accuracy * 100:.2f}%"
        )

In [ ]:
mnist_cross_norm_df = pd.DataFrame(
    mnist_cross_norm_results
)

mnist_cross_norm_df

mnist_cross_norm_matrix = (
    mnist_cross_norm_df
    .pivot(
        index="model",
        columns="attack",
        values="accuracy"
    )
)

mnist_cross_norm_matrix

In [ ]:
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

mnist_cross_norm_df.to_csv(
    RESULTS_DIR / "mnist_cross_norm_results.csv",
    index=False
)

In [ ]:
MNIST_EPSILON_GRID = {
    "linf": [
        0.00,
        0.05,
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
        0.70,
        1.00,
    ],

    "l2": [
        0.0,
        0.5,
        1.0,
        2.0,
        3.0,
        4.0,
        5.0,
        7.0,
        10.0,
    ],
}

In [ ]:
def linf_test_alpha(epsilon):
    if epsilon == 0:
        return 0.0

    return max(epsilon / 10, 1 / 255)


def l2_test_alpha(epsilon):
    if epsilon == 0:
        return 0.0

    return epsilon / 10

In [ ]:
def run_epsilon_sweep(
    models,
    loader,
    attack_fn,
    attack_name,
    epsilons,
    alpha_fn,
    steps,
    device,
    restarts=1
):
    results = []

    for model_name, model in models.items():

        for epsilon in epsilons:

            if epsilon == 0:
                _, accuracy = evaluate_clean(
                    model=model,
                    loader=loader,
                    criterion=criterion,
                    device=device
                )

            else:
                attack_kwargs = {
                    "epsilon": epsilon,
                    "alpha": alpha_fn(epsilon),
                    "steps": steps,
                    "random_start": True,
                }

                accuracy = evaluate_under_attack(
                    model=model,
                    loader=loader,
                    attack_fn=attack_fn,
                    attack_kwargs=attack_kwargs,
                    device=device,
                    restarts=restarts
                )

            results.append({
                "model": model_name,
                "attack": attack_name,
                "epsilon": epsilon,
                "accuracy": accuracy,
            })

            print(
                f"{model_name:8s} | "
                f"{attack_name:8s} | "
                f"epsilon={epsilon:.4f} | "
                f"accuracy={accuracy * 100:.2f}%"
            )

    return pd.DataFrame(results)

In [ ]:
mnist_linf_sweep_df = run_epsilon_sweep(
    models=mnist_models,
    loader=mnist_test_loader,
    attack_fn=pgd_linf_attack,
    attack_name="pgd_linf",
    epsilons=MNIST_EPSILON_GRID["linf"],
    alpha_fn=linf_test_alpha,
    steps=50,
    device=device,
    restarts=1
)

In [ ]:
mnist_l2_sweep_df = run_epsilon_sweep(
    models=mnist_models,
    loader=mnist_test_loader,
    attack_fn=pgd_l2_attack,
    attack_name="pgd_l2",
    epsilons=MNIST_EPSILON_GRID["l2"],
    alpha_fn=l2_test_alpha,
    steps=50,
    device=device,
    restarts=1
)

In [ ]:
plt.figure(figsize=(8, 5))

for model_name in mnist_linf_sweep_df["model"].unique():
    subset = mnist_linf_sweep_df[
        mnist_linf_sweep_df["model"] == model_name
    ]

    plt.plot(
        subset["epsilon"],
        subset["accuracy"],
        marker="o",
        label=model_name
    )

plt.xlabel("L_inf epsilon")
plt.ylabel("Accuracy")
plt.title("MNIST Robustness under PGD L_inf")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for model_name in mnist_l2_sweep_df["model"].unique():
    subset = mnist_l2_sweep_df[
        mnist_l2_sweep_df["model"] == model_name
    ]

    plt.plot(
        subset["epsilon"],
        subset["accuracy"],
        marker="o",
        label=model_name
    )

plt.xlabel("L2 epsilon")
plt.ylabel("Accuracy")
plt.title("MNIST Robustness under PGD L2")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
mnist_linf_sweep_df.to_csv(
    RESULTS_DIR / "mnist_pgd_linf_epsilon_sweep.csv",
    index=False
)

mnist_l2_sweep_df.to_csv(
    RESULTS_DIR / "mnist_pgd_l2_epsilon_sweep.csv",
    index=False
)